To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

### Unsloth

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported  # FastVisionModel for VLMs
import torch

max_seq_length = 2048  # Choose any! Qwen3.5 supports long context; 2048 is plenty here.

# Pick ONE dtype and use it for BOTH loading and training so they never mismatch.
# (A dtype mismatch shows up as: "mat1 and mat2 ... BFloat16 != Half".)
dtype = torch.bfloat16 if is_bfloat16_supported() else torch.float16

# Text models we support. More at https://huggingface.co/unsloth
fourbit_models = [
    "unsloth/Qwen3.5-0.8B",
    "unsloth/Qwen3.5-2B",
    "unsloth/Qwen3.5-4B",
    "unsloth/Qwen3.5-9B",
    "unsloth/Qwen3-4B",
    "unsloth/Qwen3-8B",
    "unsloth/Llama-3.2-3B-Instruct",
    "unsloth/Llama-3.1-8B-Instruct",
]  # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-4B",
    max_seq_length = max_seq_length,
    dtype = dtype,         # bf16 on Ampere+ (A100/L4/etc.), fp16 on T4
    load_in_4bit = False,  # 4-bit QLoRA is NOT recommended for Qwen3.5 - use bf16/16-bit LoRA.
    use_gradient_checkpointing = "unsloth",  # True or "unsloth" for long context
)

# Qwen3.5 is a unified vision-language model, so `tokenizer` is actually a multimodal
# *processor*. Its apply_chat_template expects each message's `content` to be a list of
# typed parts (e.g. {"type": "text", ...}) and crashes on plain-string content with
# "string indices must be integers". Our dataset is text-only, so we grab the underlying
# text tokenizer, which renders string content correctly and works directly with
# SFTTrainer and train_on_responses_only.
tok = getattr(tokenizer, "tokenizer", tokenizer)
if getattr(tok, "chat_template", None) is None:
    tok.chat_template = getattr(tokenizer, "chat_template", None)


We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

Because we are finetuning on a **text-only** dataset, we attach LoRA adapters to the **language** layers only (the attention and MLP projections). Qwen3.5 is a unified vision-language model, but we are not touching the vision encoder here. You can still choose to finetune only the attention layers, only the MLP layers, or both.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,           # The larger, the higher the accuracy, but might overfit
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention layers
        "gate_proj", "up_proj", "down_proj",     # MLP layers
    ],
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth",  # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)


<a name="Data"></a>
### Data Prep
We use the [`Cognitive-Machines-Lab/a1`](https://huggingface.co/datasets/Cognitive-Machines-Lab/a1) dataset, which is a **text-only**, multi-turn conversational dataset in ChatML format. Each row has a `messages` list containing a `system` prompt followed by alternating `user` / `assistant` turns.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("Cognitive-Machines-Lab/a1", split = "train")


Let's take an overview look at the dataset. We'll inspect the structure of one conversation and the roles inside it.

In [ ]:
dataset


In [ ]:
# This is a text-only dataset, so there are no images - each row is a conversation.
example = dataset[2]["messages"]
print("Number of turns:", len(example))
print("Roles:", [turn["role"] for turn in example])


In [ ]:
dataset[2]["messages"]


We can render a conversation exactly how the model will see it using the tokenizer's chat template:

In [ ]:
print(tok.apply_chat_template(dataset[2]["messages"], tokenize = False))


The dataset is already in the conversational ("ChatML") format that Unsloth expects, so no restructuring is needed:

```python
{ "messages" : [
    { "role": "system",    "content": "..." },
    { "role": "user",      "content": "..." },
    { "role": "assistant", "content": "..." },
    ...
] }
```

All we need to do is turn each conversation into a single `text` field by applying the chat template.

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tok.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False,  # full conversations, not a prompt
        )
        for convo in convos
    ]
    return { "text" : texts }
pass


Let's apply the formatting function to add a `text` column to every conversation in the dataset:

In [ ]:
dataset = dataset.map(formatting_prompts_func, batched = True)

# For training we only need the rendered "text" column. Dropping the raw "messages" /
# "format" columns prevents TRL from auto-detecting a "conversational" dataset and trying
# to re-apply a chat template on top of the text we already rendered. We keep `dataset`
# (with "messages") around for the inference examples below.
train_dataset = dataset.remove_columns(
    [c for c in dataset.column_names if c != "text"]
)


We look at how the first conversation is structured after formatting:

In [ ]:
dataset[0]["text"]


Let's first see, before we do any finetuning, what the base model outputs when we give it one of the system prompts and an opening line from the dataset.

In [ ]:
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

# Take the system prompt + first user turn from a dataset example as a quick test.
messages = dataset[0]["messages"][:2]  # [system, first user message]

prompt = tok.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,  # Must add for generation
)
inputs = tok(prompt, return_tensors = "pt", add_special_tokens = False).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tok, skip_prompt = True)
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    use_cache = True,
    do_sample = True,
    temperature = 0.7, top_p = 0.8, top_k = 20,
)


<a name="Train"></a>
### Train the model
Now let's train our model. We do 30 steps to speed things up, but you can set `num_train_epochs = 1` for a full run (and set `max_steps = None`). Unsloth also supports `DPOTrainer` and `GRPOTrainer` for reinforcement learning.

We use TRL's `SFTTrainer` on the rendered `text` column, then wrap it with `train_on_responses_only` so the loss is computed **only on the assistant turns** (the user / system text is masked out). This is what teaches the model to stay in character without memorizing the prompts.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)  # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tok,
    train_dataset = train_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",   # Use this for WandB etc
        # Pin the trainer precision to match the dtype Unsloth loaded the weights in,
        # otherwise you can hit a dtype mismatch ("mat1 and mat2 ..." BFloat16 != Half).
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        max_length = max_seq_length,
        packing = False,      # Keep False so train_on_responses_only works correctly
        dataset_num_proc = 1,
    ),
)

# Train ONLY on the assistant responses, ignoring the loss on the user / system turns.
# These markers match Qwen3.5's ChatML chat template.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the finetuned model! You can change the system prompt and the user message below.

Qwen3.5 uses the ChatML format, so we build the prompt with the chat template and add the generation prompt. We use Qwen3.5's recommended non-thinking sampling settings: `temperature = 0.7`, `top_p = 0.8`, `top_k = 20`.

In [ ]:
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

# Re-use a system prompt from the dataset and ask something new.
system_prompt = dataset[5]["messages"][0]["content"]
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "Are you still there? What do you remember?"},
]

prompt = tok.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,  # Must add for generation
)
inputs = tok(prompt, return_tensors = "pt", add_special_tokens = False).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tok, skip_prompt = True)
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    use_cache = True,
    do_sample = True,
    temperature = 0.7, top_p = 0.8, top_k = 20,
)


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("qwen_lora")  # Local saving
tokenizer.save_pretrained("qwen_lora")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_lora",  # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = False,      # Set to False for 16bit LoRA
    )
    FastLanguageModel.for_inference(model)  # Enable native 2x faster inference
    tok = getattr(tokenizer, "tokenizer", tokenizer)
    if getattr(tok, "chat_template", None) is None:
        tok.chat_template = getattr(tokenizer, "chat_template", None)

messages = [
    {"role": "system", "content": dataset[0]["messages"][0]["content"]},
    {"role": "user",   "content": "Hey, hey, can you hear me?"},
]

prompt = tok.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,  # Must add for generation
)
inputs = tok(prompt, return_tensors = "pt", add_special_tokens = False).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tok, skip_prompt = True)
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    use_cache = True,
    do_sample = True,
    temperature = 0.7, top_p = 0.8, top_k = 20,
)


### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Select ONLY 1 to save! (Both not needed!)

# Save locally to 16bit
if False: model.save_pretrained_merged("unsloth_finetune", tokenizer,)

# To export and save to your Hugging Face account
if False: model.push_to_hub_merged("YOUR_USERNAME/unsloth_finetune", tokenizer, token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>